# yolo26x 学習 + 疑似ラベル
1024×1024データセット + 疑似ラベル込みで22epochs学習

In [ ]:
!pip install ensemble-boxes ultralytics -q

import os
import json
import numpy as np
import pandas as pd
import torch
import torchvision.transforms.functional as TF
import shutil
import yaml
import re
from PIL import Image as PILImage
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion
from pathlib import Path
from tqdm import tqdm

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし')

DATA_DIR    = '/kaggle/input/datasets/kenkagkag/sinkai-kg/dataset1024'
PSEUDO_JSON = '/kaggle/input/datasets/kenkagkag/sinkai-kg/train_pseudo.json'
TRAIN_IMG   = f'{DATA_DIR}/images/train'
TEST_IMG    = f'{DATA_DIR}/images/test'
TRAIN_LBL   = f'{DATA_DIR}/labels/train'
YOLOX_PATH  = '/kaggle/working/runs/yolox/weights/best.pt'

# パス確認
for k, v in {
    'DATA_DIR':    DATA_DIR,
    'PSEUDO_JSON': PSEUDO_JSON,
    'TRAIN_IMG':   TRAIN_IMG,
    'TEST_IMG':    TEST_IMG,
    'TRAIN_LBL':   TRAIN_LBL,
}.items():
    print(f'{k:20s}: {os.path.exists(v)}')

# データ読み込み
if os.path.exists(PSEUDO_JSON):
    print('疑似ラベルJSONを使用')
    with open(PSEUDO_JSON) as f:
        train_data = json.load(f)
else:
    print('通常JSONを使用')
    with open(f'{DATA_DIR}/train_dataset.json') as f:
        train_data = json.load(f)

with open(f'{DATA_DIR}/test_dataset.json') as f:
    test_data = json.load(f)

cat_ids = sorted([c['id'] for c in train_data['categories']])
test_cat_ids = sorted([c['id'] for c in test_data['categories']])
assert cat_ids == test_cat_ids, 'カテゴリ不一致'

category_to_yolo = {cat_id: i for i, cat_id in enumerate(cat_ids)}
yolo_to_category = cat_ids
cat_names = {c['id']: c['name'] for c in train_data['categories']}

fname_to_id    = {os.path.basename(img['file_name']): img['id'] for img in test_data['images']}
img_id_to_info = {img['id']: img for img in test_data['images']}
test_files = sorted(os.listdir(TEST_IMG))

print(f'train画像数（pseudo込み）: {len(train_data["images"])}')
print(f'test画像数: {len(test_files)}')

In [ ]:
# データ準備（GroupKFold + 疑似ラベル）
from sklearn.model_selection import GroupKFold

def make_group_id(img):
    raw_date = img.get('date_captured', None)
    if raw_date is not None:
        try:
            dt = pd.to_datetime(str(raw_date), errors='coerce')
            date_part = dt.strftime('%Y-%m-%d') if pd.notnull(dt) else 'nodate'
        except:
            date_part = 'nodate'
    else:
        date_part = 'nodate'
    stem = Path(img.get('file_name', '')).stem.lower()
    norm = re.sub(r'[^a-z0-9]+', '_', stem)
    prefix = re.sub(r'(_?\d+)$', '', norm).strip('_')
    if len(prefix) < 3:
        prefix = norm if norm else 'nofile'
    return f'{date_part}__{prefix}'

orig_images   = [img for img in train_data['images'] if img['id'] <= 6463]
pseudo_images = [img for img in train_data['images'] if img['id'] > 6463]

groups = [make_group_id(img) for img in orig_images]
kf = GroupKFold(n_splits=5)
splits = list(kf.split(np.arange(len(orig_images)), groups=groups))
idx_tr, idx_va = splits[0]
print(f'元train: {len(idx_tr)}枚, val: {len(idx_va)}枚, 疑似ラベル: {len(pseudo_images)}枚')

WORK = '/kaggle/working/dataset'
for d in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(f'{WORK}/{d}', exist_ok=True)

ann_map = {}
for ann in train_data['annotations']:
    ann_map.setdefault(ann['image_id'], []).append(ann)

def setup_split(images, split_name):
    for img in tqdm(images, desc=split_name):
        fname = img['file_name']
        stem  = Path(fname).stem
        w, h  = img['width'], img['height']

        src_img = f'{TRAIN_IMG}/{stem}.jpg'
        if not os.path.exists(src_img):
            src_img = f'{TEST_IMG}/{stem}.jpg'
        if not os.path.exists(src_img):
            src_img = f'{TEST_IMG}/{stem}.png'
        if not os.path.exists(src_img):
            continue

        dst_img = f'{WORK}/images/{split_name}/{stem}.jpg'
        if not os.path.exists(dst_img):
            try:
                os.symlink(src_img, dst_img)
            except:
                shutil.copy(src_img, dst_img)

        src_lbl = f'{TRAIN_LBL}/{stem}.txt'
        dst_lbl = f'{WORK}/labels/{split_name}/{stem}.txt'
        if not os.path.exists(dst_lbl):
            if os.path.exists(src_lbl):
                try:
                    os.symlink(src_lbl, dst_lbl)
                except:
                    shutil.copy(src_lbl, dst_lbl)
            else:
                lines = []
                for ann in ann_map.get(img['id'], []):
                    cls = category_to_yolo[ann['category_id']]
                    x, y, bw, bh = ann['bbox']
                    cx = (x + bw / 2) / w
                    cy = (y + bh / 2) / h
                    bw /= w
                    bh /= h
                    lines.append(f'{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
                with open(dst_lbl, 'w') as f:
                    f.write('\n'.join(lines))

train_images = [orig_images[i] for i in idx_tr] + pseudo_images
val_images   = [orig_images[i] for i in idx_va]

setup_split(train_images, 'train')
setup_split(val_images,   'val')

dataset_config = {
    'path': WORK,
    'train': 'images/train',
    'val': 'images/val',
    'names': {i: cat_names[cat_id] for i, cat_id in enumerate(cat_ids)}
}
with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(dataset_config, f, allow_unicode=True)

print(f'train: {len(os.listdir(f"{WORK}/images/train"))}枚')
print(f'val:   {len(os.listdir(f"{WORK}/images/val"))}枚')

In [ ]:
# yolo26x 学習
if os.path.exists(YOLOX_PATH):
    print('学習済みyolo26xが見つかりました。スキップします。')
else:
    print('yolo26xの学習を開始します...')
    model_train = YOLO('yolo26x.pt')
    model_train.train(
        data='/kaggle/working/data.yaml',
        epochs=22,
        imgsz=1024,
        batch=4,
        hsv_v=0.7,
        val=True,
        project='/kaggle/working/runs',
        name='yolox',
        exist_ok=True,
    )
    print('yolo26x学習完了！')
    del model_train
    torch.cuda.empty_cache()

In [ ]:
# CVスコア確認
results_csv = pd.read_csv('/kaggle/working/runs/yolox/results.csv')
print(results_csv[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].tail(10))
best_epoch = results_csv['metrics/mAP50-95(B)'].idxmax()
print(f'ベストepoch: {best_epoch+1}')
print(results_csv.iloc[best_epoch][['metrics/mAP50(B)', 'metrics/mAP50-95(B)']])

In [ ]:
# 推論
model_yolox = YOLO(YOLOX_PATH)
print('yolo26x loaded')

CONF             = 0.001
IOU              = 0.6
MAX_DET          = 1000
WBF_IOU          = 0.55
WBF_SKIP_BOX_THR = 0.005

TTA_PATTERNS = [
    (False, 1024),
    (True,  1024),
    (False,  896),
]

def predict_with_tta(model, img_path, conf, iou):
    img_orig = PILImage.open(img_path).convert('RGB')
    all_boxes, all_scores, all_labels = [], [], []
    for do_flip, sz in TTA_PATTERNS:
        img = img_orig.copy()
        if do_flip:
            img = TF.hflip(img)
        result = model.predict(
            source=img, conf=conf, iou=iou,
            max_det=MAX_DET, imgsz=sz,
            verbose=False, half=True,
        )[0]
        if len(result.boxes) == 0:
            continue
        boxes  = result.boxes.xyxyn.cpu().numpy().tolist()
        scores = result.boxes.conf.cpu().numpy().tolist()
        labels = result.boxes.cls.cpu().numpy().astype(int).tolist()
        if do_flip:
            boxes = [[1-x2, y1, 1-x1, y2] for x1, y1, x2, y2 in boxes]
        boxes = [[min(max(v, 0.0), 1.0) for v in box] for box in boxes]
        all_boxes.append(boxes)
        all_scores.append(scores)
        all_labels.append(labels)
    if not all_boxes:
        return [], [], []
    boxes_f, scores_f, labels_f = weighted_boxes_fusion(
        all_boxes, all_scores, all_labels,
        weights=[1.0] * len(all_boxes),
        iou_thr=WBF_IOU, skip_box_thr=WBF_SKIP_BOX_THR,
    )
    return boxes_f.tolist(), scores_f.tolist(), labels_f.tolist()


rows = []
for fname in tqdm(test_files):
    img_path = os.path.join(TEST_IMG, fname)
    image_id = fname_to_id.get(fname)
    if image_id is None:
        stem = Path(fname).stem
        for key in fname_to_id:
            if Path(key).stem == stem:
                image_id = fname_to_id[key]
                break
    if image_id is None:
        continue
    w = img_id_to_info[image_id]['width']
    h = img_id_to_info[image_id]['height']
    boxes_f, scores_f, labels_f = predict_with_tta(model_yolox, img_path, CONF, IOU)
    if len(boxes_f) == 0:
        continue
    for box, score, label in zip(boxes_f, scores_f, labels_f):
        x1, y1, x2, y2 = box
        rows.append({
            'image_id':    image_id,
            'category_id': yolo_to_category[int(label)],
            'bbox_x':      x1 * w,
            'bbox_y':      y1 * h,
            'bbox_width':  (x2 - x1) * w,
            'bbox_height': (y2 - y1) * h,
            'score':       float(score),
        })

print(f'予測数: {len(rows)}')
print(f'1画像あたり平均: {len(rows)/len(test_files):.1f}box')

In [ ]:
# 提出ファイル作成
submission = pd.DataFrame(rows)
submission['annotation_id'] = np.arange(len(submission))
submission = submission[[
    'annotation_id', 'image_id', 'category_id',
    'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height', 'score'
]]
submission['score'] = submission['score'].clip(0, 1)
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f'完了！{len(submission)}行')
print(submission.head())